# Data Vortex - Round 1: Data Cleaning & Validation

## 1. Objective and Cleaning Rules
This notebook implements the complete, reproducible data cleaning pipeline for the **Data Vortex Round 1** competition.

### Strict Governance Constraints:
- **Raw Datasets Untouched:** `Social_Engine_Users.csv` and `Social_Engine_Posts_Corrupted.csv` remain strictly read-only.
- **Clean Datasets Generated:** Saved exclusively to `data/cleaned/`.
- **No Data Fabrication:** Missing values are NOT imputed. Synthetic distributions are NOT altered.
- **Only Justified Transformations:**
  1. Remove 360 exact duplicate rows from Posts (yielding exactly 12,000 unique rows).
  2. Rectify negative likes via `abs(likes)` (correcting 509 sign-flipped values).
  3. Standardize timestamps into unified `YYYY-MM-DD HH:MM:SS` (handling ISO 8601, Unix epoch seconds, DD-MM-YYYY).
  4. Trim leading and trailing whitespace on `text_content`.
  5. Decode literal HTML entity `&amp;` to `&` in `text_content`.
  6. Preserve missing values as `NaN`/null without imputation.

In [ ]:
import os
import re
import pandas as pd
import numpy as np

RAW_USERS = os.path.join("..", "data", "raw", "Social_Engine_Users.csv")
RAW_POSTS = os.path.join("..", "data", "raw", "Social_Engine_Posts_Corrupted.csv")
CLEAN_USERS = os.path.join("..", "data", "cleaned", "Social_Engine_Users_Cleaned.csv")
CLEAN_POSTS = os.path.join("..", "data", "cleaned", "Social_Engine_Posts_Cleaned.csv")

print(f"Raw Users Exists: {os.path.exists(RAW_USERS)}")
print(f"Raw Posts Exists: {os.path.exists(RAW_POSTS)}")

## 2. Clean Users Dataset
- Preserve all 1,500 rows and 5 columns.
- Preserve `user_id`, `location` (including `Singapore` and `São Paulo, Brazil`), `language`, and `follower_count`.
- Standardize `account_created` to ISO format `YYYY-MM-DD`.
- Output clean dataset with UTF-8 encoding.

In [ ]:
df_users = pd.read_csv(RAW_USERS, encoding="utf-8")
print(f"Raw Users shape: {df_users.shape}")

# Format account_created to standard YYYY-MM-DD
df_users["account_created"] = pd.to_datetime(df_users["account_created"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d")
df_users["follower_count"] = df_users["follower_count"].astype("int64")

# Validation assertions
assert len(df_users) == 1500, "Users row count must be exactly 1,500"
assert df_users["user_id"].nunique() == 1500, "Users IDs must be 100% unique"
assert df_users.isnull().sum().sum() == 0, "Users must have 0 nulls"
assert (df_users["location"] == "Singapore").sum() == 49, "Singapore count must be preserved"
assert (df_users["location"] == "São Paulo, Brazil").sum() == 44, "São Paulo count must be preserved"

# Save clean Users CSV
os.makedirs(os.path.dirname(CLEAN_USERS), exist_ok=True)
df_users.to_csv(CLEAN_USERS, index=False, encoding="utf-8")
print(f"Clean Users saved successfully: {CLEAN_USERS} ({len(df_users)} rows)")

## 3. Clean Posts Dataset

### Step 3.1: Exact Duplicate Removal
Remove the 360 exact duplicate rows, reducing the dataset from 12,360 to exactly 12,000 unique records.

In [ ]:
df_posts_raw = pd.read_csv(RAW_POSTS, encoding="utf-8")
print(f"Raw Posts Shape: {df_posts_raw.shape}")

duplicate_count = df_posts_raw.duplicated().sum()
print(f"Exact Duplicate Rows Identified: {duplicate_count:,}")

df_posts_clean = df_posts_raw.drop_duplicates().copy()
print(f"Posts Shape after Exact Deduplication: {df_posts_clean.shape}")
assert len(df_posts_clean) == 12000, f"Expected 12,000 rows, got {len(df_posts_clean)}"
assert df_posts_clean["post_id"].nunique() == 12000, "All post_ids must be unique"

### Step 3.2: Rectify Negative Likes (Sign Inversion)
Convert negative likes to positive integers using `abs()`. Missing values remain missing.

In [ ]:
neg_likes_before = (df_posts_clean["likes"] < 0).sum()
null_likes_before = df_posts_clean["likes"].isnull().sum()
print(f"Negative Likes Before Fix: {neg_likes_before:,}")
print(f"Missing Likes Before Fix: {null_likes_before:,}")

df_posts_clean["likes"] = df_posts_clean["likes"].apply(lambda x: abs(x) if pd.notnull(x) else x)
df_posts_clean["likes"] = df_posts_clean["likes"].astype("Int64")

neg_likes_after = (df_posts_clean["likes"] < 0).sum()
null_likes_after = df_posts_clean["likes"].isnull().sum()
print(f"Negative Likes After Fix: {neg_likes_after}")
print(f"Missing Likes After Fix: {null_likes_after:,}")
assert neg_likes_after == 0, "Zero negative likes must remain"

### Step 3.3: Standardize Timestamps
Standardize heterogeneous formats (`ISO 8601`, `10-digit Unix epoch seconds`, `DD-MM-YYYY`) into `YYYY-MM-DD HH:MM:SS`.

In [ ]:
def standardize_ts(ts_val):
    val_str = str(ts_val).strip()
    if re.match(r"^\d{10}$", val_str):
        return pd.to_datetime(int(val_str), unit="s").strftime("%Y-%m-%d %H:%M:%S")
    if re.match(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}", val_str):
        return pd.to_datetime(val_str).strftime("%Y-%m-%d %H:%M:%S")
    if re.match(r"^\d{2}-\d{2}-\d{4}$", val_str):
        return pd.to_datetime(val_str, format="%d-%m-%Y").strftime("%Y-%m-%d %H:%M:%S")
    raise ValueError(f"Unrecognized timestamp: {ts_val}")

df_posts_clean["timestamp"] = df_posts_clean["timestamp"].apply(standardize_ts)

ts_dt = pd.to_datetime(df_posts_clean["timestamp"])
print(f"Earliest Cleaned Timestamp: {ts_dt.min()}")
print(f"Latest Cleaned Timestamp: {ts_dt.max()}")
assert ts_dt.isnull().sum() == 0, "Zero unparseable timestamps"
assert ts_dt.min() >= pd.Timestamp("2024-05-01 00:00:00"), "Date must be on or after 2024-05-01"
assert ts_dt.max() <= pd.Timestamp("2025-04-30 23:59:59"), "Date must be on or before 2025-04-30"

### Step 3.4 & 3.5: Clean Text Whitespace & Decode HTML Entity `&amp;`
- Trim leading/trailing whitespace without modifying internal spacing.
- Replace literal `&amp;` with `&`.

In [ ]:
def clean_post_text(text):
    if pd.isna(text):
        return text
    cleaned = str(text).strip()
    cleaned = cleaned.replace("&amp;", "&")
    return cleaned

padded_before = df_posts_clean["text_content"].dropna().apply(lambda x: x != x.strip()).sum()
amp_before = df_posts_clean["text_content"].dropna().str.contains("&amp;").sum()
print(f"Padded text rows before: {padded_before:,}")
print(f"Rows with &amp; before: {amp_before:,}")

df_posts_clean["text_content"] = df_posts_clean["text_content"].apply(clean_post_text)

padded_after = df_posts_clean["text_content"].dropna().apply(lambda x: x != x.strip()).sum()
amp_after = df_posts_clean["text_content"].dropna().str.contains("&amp;").sum()
print(f"Padded text rows after: {padded_after}")
print(f"Rows with &amp; after: {amp_after}")
assert padded_after == 0, "Zero padded text strings must remain"
assert amp_after == 0, "Zero raw &amp; entities must remain"

### Step 3.6: Preserve Other Columns & Save Clean Posts CSV
- `shares` and `comments` remain unchanged.
- Missing values are preserved as nulls (empty fields in CSV).
- Write to `Social_Engine_Posts_Cleaned.csv` with UTF-8 encoding.

In [ ]:
# Validate referential integrity against clean Users
valid_users = set(df_users["user_id"])
orphan_count = (~df_posts_clean["user_id"].isin(valid_users)).sum()
print(f"Orphan posts count: {orphan_count}")
assert orphan_count == 0, "Zero orphan posts permitted"

# Save clean Posts CSV
df_posts_clean.to_csv(CLEAN_POSTS, index=False, encoding="utf-8")
print(f"Clean Posts saved successfully: {CLEAN_POSTS} ({len(df_posts_clean):,} rows)")

## 4. Final Validation & Comparative Audit
Compare the raw and cleaned datasets to verify exact compliance with all competition requirements.

In [ ]:
clean_u = pd.read_csv(CLEAN_USERS)
clean_p = pd.read_csv(CLEAN_POSTS)

validation_summary = pd.DataFrame([
    {"Audit Metric": "Users Row Count", "Raw Expected": "1,500", "Cleaned Verified": f"{len(clean_u):,}", "Status": "PASS"},
    {"Audit Metric": "Users Unique IDs", "Raw Expected": "1,500", "Cleaned Verified": f"{clean_u['user_id'].nunique():,}", "Status": "PASS"},
    {"Audit Metric": "Users Missing Values", "Raw Expected": "0", "Cleaned Verified": f"{clean_u.isnull().sum().sum()}", "Status": "PASS"},
    {"Audit Metric": "Posts Row Count", "Raw Expected": "12,360", "Cleaned Verified": f"{len(clean_p):,}", "Status": "PASS (360 dups removed)"},
    {"Audit Metric": "Posts Unique Post IDs", "Raw Expected": "12,000", "Cleaned Verified": f"{clean_p['post_id'].nunique():,}", "Status": "PASS"},
    {"Audit Metric": "Posts Exact Duplicates", "Raw Expected": "360", "Cleaned Verified": f"{clean_p.duplicated().sum()}", "Status": "PASS"},
    {"Audit Metric": "Posts Negative Likes", "Raw Expected": "525 (509 dedup)", "Cleaned Verified": f"{(clean_p['likes'] < 0).sum()}", "Status": "PASS"},
    {"Audit Metric": "Posts Orphan User IDs", "Raw Expected": "0", "Cleaned Verified": f"{(~clean_p['user_id'].isin(clean_u['user_id'])).sum()}", "Status": "PASS"},
    {"Audit Metric": "Posts Unparseable Timestamps", "Raw Expected": "0", "Cleaned Verified": f"{clean_p['timestamp'].isnull().sum()}", "Status": "PASS"},
    {"Audit Metric": "Posts Padded Text Content", "Raw Expected": "337 (329 dedup)", "Cleaned Verified": f"{(clean_p['text_content'].dropna().apply(lambda x: x != x.strip())).sum()}", "Status": "PASS"},
    {"Audit Metric": "Posts HTML Entities (&amp;)", "Raw Expected": "341 (328 dedup)", "Cleaned Verified": f"{clean_p['text_content'].dropna().str.contains('&amp;').sum()}", "Status": "PASS"}
])

validation_summary